<a href="https://colab.research.google.com/github/nasif-raihan/GSU/blob/main/CSCI_7090_Module_8_Homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 8 — Homework
**Computer Vision · Residual Networks (ResNet)**

Your block parameter, ResNet version, channel count, and stride are generated from your **roll number**, so your numbers are unique — pasted generic answers will not match your outputs.

## Rules
- Set `ROLL_NUMBER` at the top, then run the setup cell. Do **not** edit the parameter lines.
- Each question is a **task**: write the code yourself and store the result in the named variable(s) under **Produce**. There are no fill-in-the-blank stubs.
- Answer the **Written** prompts in your own words. Run the final **Submission Summary** and keep its output.

## Grading (100): Q1 10 · Q2 15 · Q3 10 · Q4 10 · Q5 10 · Q6 25 · Q7 20  ·  ~2 hours (use a GPU runtime)

## Setup — run first
Loads **CIFAR-10** into `xtr, ytr, xte, yte` and generates your personal parameters. A helper `resnet_layer(x, num_filters, ...)` (Conv → BatchNorm → ReLU) is provided.

In [1]:
# === SETUP — run first ===
ROLL_NUMBER = "901503252"
import hashlib, numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, Add, Input, GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model

seed = int(hashlib.sha256(ROLL_NUMBER.encode()).hexdigest(), 16) % (2**32)
rng  = np.random.default_rng(seed)
tf.random.set_seed(seed)

# CIFAR-10 (one-line import) for the training task in Q6
(xtr, ytr), (xte, yte) = tf.keras.datasets.cifar10.load_data()
xtr = xtr.astype("float32") / 255.0; xte = xte.astype("float32") / 255.0
ytr = ytr.flatten(); yte = yte.flatten()

# ---- Your personal parameters (unique to your roll number) — DO NOT EDIT ----
N        = int(rng.choice([2, 3, 5, 7]))       # blocks-per-stack parameter
VERSION  = int(rng.choice([1, 2]))             # ResNet v1 (6n+2) or v2 (9n+2)
FILTERS  = int(rng.choice([16, 32, 64]))       # channels in the block
STRIDES  = int(rng.choice([1, 2]))             # stride of the block

def resnet_layer(x, num_filters, kernel_size=3, strides=1, activation="relu"):
    x = Conv2D(num_filters, kernel_size, strides=strides, padding="same", kernel_initializer="he_normal")(x)
    x = BatchNormalization()(x)
    if activation: x = Activation(activation)(x)
    return x

print("Roll:", ROLL_NUMBER, "| seed:", seed)
print(f"N={N}  VERSION={VERSION}  FILTERS={FILTERS}  STRIDES={STRIDES}")
print("TIP: use a GPU runtime (Runtime -> Change runtime type -> GPU) so Q6 trains quickly.")

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 1977s 12us/step
Roll: 901503252 | seed: 279384723
N=2  VERSION=2  FILTERS=16  STRIDES=2
TIP: use a GPU runtime (Runtime -> Change runtime type -> GPU) so Q6 trains quickly.


## Q1 — ResNet depth formula (10 pts)
Compute the network `depth` and the number of residual blocks per stack for **your** `N` and `VERSION` (v1: `depth = 6·N + 2`; v2: `depth = 9·N + 2`; blocks per stack `= (depth − 2) // (6 or 9)`).

**Produce:** `depth, blocks_per_stack`

In [2]:
# Compute depth and blocks_per_stack from N and VERSION.
# Produce: depth, blocks_per_stack

if VERSION == 1:
    depth = 6 * N + 2
    blocks_per_stack = (depth - 2) // 6
elif VERSION == 2:
    depth = 9 * N + 2
    blocks_per_stack = (depth - 2) // 9

print(f"depth = {depth}")
print(f"blocks_per_stack = {blocks_per_stack}")

depth = 20
blocks_per_stack = 2


**Written:** Why can ResNet be made much deeper than a plain CNN without the accuracy degrading?

> Residual networks introduce identity skip-connections that allow gradients to flow backwards directly through the network without being scaled repeatedly by weight matrices. This mitigates vanishing and exploding gradient problems during backpropagation, making deep architectures significantly easier to optimize.

## Q2 — Build an identity residual block (15 pts)
Build a residual block: two conv layers (use the `resnet_layer` helper — the **second** with `activation=None`), then **add the input back** (`H(x) = F(x) + x`), then a final ReLU. Apply it to a random `8×8×FILTERS` tensor and confirm the output shape equals the input shape.

**Produce:** the `block` model

In [3]:
# Build an identity residual block on an (8,8,FILTERS) input; apply to a random tensor.
# Produce: block

inputs = Input(shape=(8, 8, FILTERS))
x = resnet_layer(inputs, num_filters=FILTERS, strides=1, activation="relu")
x = resnet_layer(x, num_filters=FILTERS, strides=1, activation=None)
x = Add()([x, inputs])
outputs = Activation("relu")(x)

block = Model(inputs=inputs, outputs=outputs)
print("Model output shape:", block.output_shape)

Model output shape: (None, 8, 8, 16)


**Written:** Why must the skip-connection tensor and the residual branch have matching shapes before they can be added?

> Element-wise addition requires both operand tensors to have identical spatial dimensions (height and width) and the same number of filters (channels) so that corresponding elements can be correctly added together.

## Q3 — Count the block's parameters (10 pts)
Report the total number of parameters in your Q2 `block`.

**Produce:** `n_params`

In [4]:
# Report the total parameter count of block.
# Produce: n_params

n_params = block.count_params()
print("n_params =", n_params)

n_params = 4768


**Written:** ResNet uses Global Average Pooling instead of large fully connected layers before the classifier. How does that affect the parameter count?

> Global Average Pooling collapses each 2D feature map into a single scalar value. This means the input to the final Dense classifier layer is equal to the number of filters, rather than the flattened spatial area multiplied by filters. This significantly reduces the total parameter count and reduces overfitting.

## Q4 — Projection shortcut (10 pts)
When a block downsamples (`STRIDES > 1`) or changes channel count, the skip path needs a **1×1 convolution** to match shapes. Apply a `1×1` conv with **your** `STRIDES` to a `16×16×FILTERS` input and report the output shape.

**Produce:** `shortcut_shape`

In [5]:
# Apply a 1x1 Conv2D with STRIDES to a (16,16,FILTERS) input; report the output shape.
# Produce: shortcut_shape

inputs_q4 = Input(shape=(16, 16, FILTERS))
shortcut_layer = Conv2D(FILTERS, kernel_size=1, strides=STRIDES, padding="same", kernel_initializer="he_normal")(inputs_q4)
shortcut_model = Model(inputs=inputs_q4, outputs=shortcut_layer)

shortcut_shape = tuple(shortcut_model.output_shape[1:])
print("shortcut_shape =", shortcut_shape)

shortcut_shape = (8, 8, 16)


**Written:** For your `STRIDES`, does the spatial size change? When would an **identity** shortcut (no 1×1 conv) be enough instead?

> If `STRIDES=2`, the spatial size downsamples by half (from 16x16 to 8x8). If `STRIDES=1`, the spatial size remains the same (16x16). An identity shortcut is enough only when the spatial resolution and the channel depth of the input exactly match the output of the residual branch.

## Q5 — Identity vs projection rule (10 pts)
Write a function `needs_projection(strides, in_ch, out_ch)` that returns `True` if a projection shortcut is required (stride ≠ 1 **or** the channel count changes), else `False`.

**Produce:** the `needs_projection` function

In [6]:
# Write needs_projection(strides, in_ch, out_ch). Produce: needs_projection

def needs_projection(strides, in_ch, out_ch):
    return strides != 1 or in_ch != out_ch

**Written:** Explain your rule in words.

> A projection shortcut (1x1 convolution) is necessary whenever the spatial size is downsampled (strides is not 1) or the number of channels (filters) changes between the input and output.

## Q6 — Build & train a small ResNet on CIFAR-10 (25 pts)
Assemble a small ResNet for CIFAR-10 (input `32×32×3`, 10 classes): a stem conv, then **at least two** residual blocks (reuse your Q2 pattern — add a 1×1 projection on the skip path whenever channels change), then GlobalAveragePooling2D → Dense(10, softmax). Compile and train it (a subset and a few epochs is fine), then report **test accuracy**.

**Produce:** the `model` and `acc` (test accuracy)

In [7]:
# Build a small ResNet, train on a CIFAR-10 subset, evaluate.
# Produce: model, acc

inputs_resnet = Input(shape=(32, 32, 3))

# Stem
x = resnet_layer(inputs_resnet, num_filters=FILTERS, strides=1)

# Block 1
in_ch1 = FILTERS
out_ch1 = FILTERS
shortcut1 = x
y1 = resnet_layer(x, num_filters=out_ch1, strides=1, activation="relu")
y1 = resnet_layer(y1, num_filters=out_ch1, strides=1, activation=None)
if needs_projection(1, in_ch1, out_ch1):
    shortcut1 = Conv2D(out_ch1, kernel_size=1, strides=1, padding="same")(shortcut1)
x = Add()([y1, shortcut1])
x = Activation("relu")(x)

# Block 2
in_ch2 = FILTERS
out_ch2 = FILTERS * 2
shortcut2 = x
y2 = resnet_layer(x, num_filters=out_ch2, strides=STRIDES, activation="relu")
y2 = resnet_layer(y2, num_filters=out_ch2, strides=1, activation=None)
if needs_projection(STRIDES, in_ch2, out_ch2):
    shortcut2 = Conv2D(out_ch2, kernel_size=1, strides=STRIDES, padding="same")(shortcut2)
x = Add()([y2, shortcut2])
x = Activation("relu")(x)

x = GlobalAveragePooling2D()(x)
outputs_resnet = Dense(10, activation="softmax")(x)

model = Model(inputs=inputs_resnet, outputs=outputs_resnet)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# Fast training on a subset
model.fit(xtr[:5000], ytr[:5000], batch_size=64, epochs=3, validation_split=0.1, verbose=1)
loss, acc = model.evaluate(xte[:1000], yte[:1000], verbose=0)
print("Test Accuracy:", acc)

Epoch 1/3
71/71 ━━━━━━━━━━━━━━━━━━━━ 21s 133ms/step - accuracy: 0.2396 - loss: 2.0876 - val_accuracy: 0.1340 - val_loss: 2.1904
Epoch 2/3
71/71 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.3207 - loss: 1.8471 - val_accuracy: 0.2460 - val_loss: 2.0732
Epoch 3/3
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.3756 - loss: 1.7240 - val_accuracy: 0.2640 - val_loss: 1.9808
Test Accuracy: 0.27300000190734863


**Written:** Did you add a projection shortcut anywhere in your ResNet? At which block, and why was it needed there?

> Yes, a projection shortcut was added in Block 2. It was required there because we increased the filters from `FILTERS` to `FILTERS * 2` (causing a mismatch in channel size) and used `STRIDES` which downsamples the spatial size.

**Written:** Answer in your own words:

**(a)** The **degradation problem** is the drop in accuracy when extremely deep plain networks are trained, which is not caused by overfitting. A residual skip connection solves this by introducing shortcut paths that bypass layers, allowing gradients to propagate directly.

**(b)** Because of the additive identity term in `dH/dx = dF/dx + 1`, even if the weights of the intermediate layers shrink and the gradient through the path `dF/dx` vanishes, the total gradient retains at least the `1` term. This prevents gradient vanishing during backpropagation.

**(c)** An **identity** shortcut is enough when the input and output have exactly identical spatial resolution and channel depth. A **projection** shortcut is needed when there is a change in resolution (strides > 1) or channel depth (filters change).

## Submission Summary
Run last; keep the output.

In [8]:
# === SUBMISSION SUMMARY ===
print("Roll:", ROLL_NUMBER, "| seed:", seed)
print("Q1 depth/blocks:", depth, blocks_per_stack, "| N,VERSION", (N, VERSION))
print("Q2 block output shape:", tuple(block(tf.random.normal((1,8,8,FILTERS))).shape))
print("Q3 block params:", n_params)
print("Q4 shortcut shape (STRIDES=%d):" % STRIDES, shortcut_shape)
print("Q5 needs_projection(your case):", needs_projection(STRIDES, FILTERS, FILTERS))
print("Q6 model params:", model.count_params(), "| test accuracy:", round(acc, 4))

Roll: 901503252 | seed: 279384723
Q1 depth/blocks: 20 2 | N,VERSION (2, 2)
Q2 block output shape: (1, 8, 8, 16)
Q3 block params: 4768
Q4 shortcut shape (STRIDES=2): (8, 8, 16)
Q5 needs_projection(your case): True
Q6 model params: 20298 | test accuracy: 0.273
